In [1]:
!pip install streamlit ultralytics opencv-python

In [49]:
code = """
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import streamlit as st
import cv2
import tempfile
from ultralytics import YOLO

st.title("AI Bag Counter system")

uploaded_file = st.file_uploader("Upload Video", type=["mp4", "avi", "mov"])

if uploaded_file is not None:

    temp_file = tempfile.NamedTemporaryFile(delete=False)
    temp_file.write(uploaded_file.read())

    model = YOLO("yolov8n.pt")

    cap = cv2.VideoCapture(temp_file.name)

    bag_count = 0
    counted_ids = set()
    previous_positions = {}

    frame_placeholder = st.empty()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (640, 480))
        height, width, _ = frame.shape

        red_line = width // 2
        buffer = 10

        results = model.track(frame, persist=True)

        if results[0].boxes.id is not None:

            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy()

            for box, track_id in zip(boxes, ids):

                x1, y1, x2, y2 = box
                center_x = int((x1 + x2) / 2)

                cv2.rectangle(frame,
                              (int(x1), int(y1)),
                              (int(x2), int(y2)),
                              (0, 255, 0), 2)

                if track_id not in previous_positions:
                    previous_positions[track_id] = center_x
                else:
                    prev_x = previous_positions[track_id]

                    # 🔥 RIGHT ➜ LEFT LINE CROSSING
                    if (
                        prev_x > red_line and
                        center_x < red_line and
                        track_id not in counted_ids
                    ):
                        bag_count += 1
                        counted_ids.add(track_id)

                    previous_positions[track_id] = center_x

        # Draw RED LINE
        cv2.line(frame,
                 (red_line, 0),
                 (red_line, height),
                 (0, 0, 255), 3)

        cv2.putText(frame,
                    f"Bag Count: {bag_count}",
                    (30, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5,
                    (0, 255, 255),
                    3)

        frame_placeholder.image(frame, channels="BGR")

    cap.release()

    st.success(f"Final Bag Count: {bag_count}")

"""


with open("app.py", "w", encoding="utf-8") as f:
    f.write(code)

print("✅ app.py saved successfully!")


✅ app.py saved successfully!
